In [1]:
!pip install Pillow

In [2]:
!pip install -q rfdetr>=1.4.0 supervision roboflow

In [3]:


import torch
import cv2
import numpy as np
import supervision as sv
from PIL import Image
from tqdm import tqdm
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

# If you have your custom module:
from rfdetr import RFDETRLarge

# ------------------------
# CONFIGURATION
# ------------------------
CHECKPOINT_PATH = "/content/drive/MyDrive/Projects/RoadEye/RFDETR_ROADEYE_Large/Models/checkpoint_best_ema.pth"
CONF_THRESHOLD = 0.4
IMG_SIZE = 640
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading models on: {DEVICE}")

# 1. LOAD SEGFORMER (Road Segmentation)
processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-cityscapes-512-1024")
seg_model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b0-finetuned-cityscapes-512-1024").to(DEVICE)
seg_model.eval()

# 2. LOAD RF-DETR (Pothole Detection)
det_model = RFDETRLarge(
    num_classes=1,
    pretrain_weights=CHECKPOINT_PATH,
    resolution=IMG_SIZE
)
det_model.optimize_for_inference()

print("✅ Both models loaded successfully!")

Loading models on: cuda


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


preprocessor_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/14.9M [00:00<?, ?B/s]

[2026-03-27 06:56:59] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-03-27 06:56:59] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
`use_return_dict` is deprecated! Use `return_dict` instead!


✅ Both models loaded successfully!


In [4]:
def get_road_mask(image: Image.Image, processor, model, device):
    """Generates a binary road mask from a PIL Image using Segformer."""
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    # Resize logits to match the original image size
    logits = torch.nn.functional.interpolate(
        logits,
        size=image.size[::-1], # PIL is (W,H), PyTorch expects (H,W)
        mode="bilinear",
        align_corners=False,
    )

    predicted_mask = logits.argmax(dim=1).squeeze().cpu().numpy()

    # Class '0' is 'road' in Cityscapes.
    # (You could also add '1' for sidewalk if needed: predicted_mask <= 1)
    road_mask = (predicted_mask == 0).astype(np.uint8)

    # Dilate mask slightly so we don't accidentally cut off potholes exactly on the edge
    kernel = np.ones((15, 15), np.uint8)
    road_mask = cv2.dilate(road_mask, kernel, iterations=1)

    return road_mask

def filter_detections_by_road(detections, road_mask, overlap_threshold=0.4):
    """Filters RF-DETR detections, keeping only those that overlap the road mask."""
    if len(detections) == 0:
        return detections

    keep_mask =[]

    for box in detections.xyxy:
        x1, y1, x2, y2 = map(int, box)

        # Keep bounds safely inside the image
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(road_mask.shape[1], x2), min(road_mask.shape[0], y2)

        box_area = (x2 - x1) * (y2 - y1)
        if box_area == 0:
            keep_mask.append(False)
            continue

        # Extract the area of the bounding box from the binary road mask
        box_mask = road_mask[y1:y2, x1:x2]
        road_pixels = np.sum(box_mask)

        # Calculate what percentage of the bounding box is ON the road
        road_ratio = road_pixels / box_area

        # Keep if enough of the box is on the road
        keep_mask.append(road_ratio >= overlap_threshold)

    return detections[np.array(keep_mask)]

#Trajectory Guidelines v1


In [ ]:
# ============================================================
# 🚧 ROI SPLIT LINE — Inference Zone Divider
# Inspired by reverse parking sensor trajectory guidelines.
# Only the LOWER half of the frame is used for detection.
# The upper half (far-field / farsight) is ignored entirely.
# ============================================================

# ----------------------------
# CONFIGURABLE SPLIT RATIO
# ----------------------------
# 0.5 = dead centre | 0.6 = lower 40% | 0.4 = lower 60%
ROI_SPLIT_RATIO = 0.5

frame_h = video_info.height
frame_w = video_info.width
SPLIT_Y = int(frame_h * ROI_SPLIT_RATIO)   # pixel row of the divider line

print(f"🔲 Frame size   : {frame_w} x {frame_h}")
print(f"📐 Split line Y : {SPLIT_Y}px  ({ROI_SPLIT_RATIO*100:.0f}% from top)")
print(f"✅ Inference zone: rows {SPLIT_Y} → {frame_h}  (lower {(1-ROI_SPLIT_RATIO)*100:.0f}% of frame)")


# ----------------------------
# HELPER — draw the divider UI
# ----------------------------
def draw_roi_divider(frame_rgb, split_y, frame_w):
    """
    Draws a reverse-parking-sensor-style boundary line on the frame.
    - Dashed horizontal line at split_y
    - Translucent dark overlay on the UPPER (ignored) zone
    - Small 'IGNORED ZONE' label on the upper half
    - Arrow-style chevrons pointing downward (like parking guidelines)
    """
    out = frame_rgb.copy()

    # ── 1. Dark overlay on the upper / ignored zone ──────────────────
    overlay = out.copy()
    cv2.rectangle(overlay, (0, 0), (frame_w, split_y), (30, 30, 30), -1)
    out = cv2.addWeighted(overlay, 0.35, out, 0.65, 0)

    # ── 2. Dashed boundary line ───────────────────────────────────────
    dash_len, gap_len = 30, 15
    x = 0
    while x < frame_w:
        x_end = min(x + dash_len, frame_w)
        cv2.line(out, (x, split_y), (x_end, split_y), (255, 220, 0), 2)
        x += dash_len + gap_len

    # ── 3. "ACTIVE ZONE" label (lower half) ──────────────────────────
    zone_label_y = split_y + 18
    cv2.putText(out, "[ ACTIVE DETECTION ZONE ]",
                (10, zone_label_y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                (255, 220, 0), 1, cv2.LINE_AA)

    # ── 4. "IGNORED" label (upper half) ──────────────────────────────
    cv2.putText(out, "[ FAR-FIELD — IGNORED ]",
                (10, max(18, split_y - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                (160, 160, 160), 1, cv2.LINE_AA)

    # ── 5. Parking-sensor chevrons (▼) centred on the split line ─────
    chevron_color = (255, 220, 0)
    chevron_h, chevron_w = 16, 24
    num_chevrons = 5
    spacing = frame_w // (num_chevrons + 1)
    for n in range(1, num_chevrons + 1):
        cx = n * spacing
        # downward-pointing triangle / chevron
        pts = np.array([
            [cx - chevron_w // 2, split_y - chevron_h // 2],
            [cx + chevron_w // 2, split_y - chevron_h // 2],
            [cx,                  split_y + chevron_h // 2]
        ], dtype=np.int32)
        cv2.fillPoly(out, [pts], chevron_color)

    return out


# ----------------------------
# HELPER — ROI-aware detection filter
# ----------------------------
def filter_detections_by_roi(detections, split_y):
    """
    Drops any detection whose bounding-box CENTRE is above split_y.
    Keeps only potholes in the lower (near-field) zone.
    """
    if len(detections) == 0:
        return detections

    keep = []
    for i, box in enumerate(detections.xyxy):
        x1, y1, x2, y2 = box
        centre_y = (y1 + y2) / 2
        if centre_y >= split_y:          # centre is in lower half → keep
            keep.append(i)

    # Rebuild a filtered Detections object
    if not keep:
        return sv.Detections.empty()

    filtered = sv.Detections(
        xyxy        = detections.xyxy[keep],
        confidence  = detections.confidence[keep] if detections.confidence is not None else None,
        class_id    = detections.class_id[keep]   if detections.class_id   is not None else None,
    )
    return filtered

print("✅ ROI helpers defined — ready for the inference loop.")

#High Quality Final


In [6]:
# ============================================================
#  R O A D E Y E  —  FULL INFERENCE PIPELINE (SINGLE CELL)
# ============================================================
#
#  DEBUG = True  → shows everything:
#                   road segmentation mask, edge overlays,
#                   ROI trajectory divider + chevrons,
#                   far-field dimming, zone labels,
#                   per-pothole edge-density & score HUD
#
#  DEBUG = False → clean output:
#                   only bounding boxes + severity label
#                   (no overlays, no divider decorations)
#
# ============================================================

import os
import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm
import supervision as sv

# ─────────────────────────────────────────
#  ① MASTER SWITCHES  ← edit here only
# ─────────────────────────────────────────
DEBUG             = True    # False = clean bounding-boxes only

# ROI split — fraction of frame height from the TOP that is ignored
# 0.5 → bottom 50%  |  0.4 → bottom 60%  |  0.6 → bottom 40%
ROI_SPLIT_RATIO   = 0.5

CONF_THRESHOLD    = 0.3
MASK_UPDATE_FREQ  = 3       # re-run road segmentation every N frames

# ─────────────────────────────────────────
#  ② PATHS
# ─────────────────────────────────────────
os.makedirs("/content/output", exist_ok=True)
SOURCE_VIDEO_PATH = "/content/IndianPotholes_Dashcam_2.mp4"
TARGET_VIDEO_PATH = "/content/output/RoadEye_result.mp4"

# ─────────────────────────────────────────
#  ③ VIDEO INFO
# ─────────────────────────────────────────
video_info  = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
generator   = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)

frame_h     = video_info.height
frame_w     = video_info.width
SPLIT_Y     = int(frame_h * ROI_SPLIT_RATIO)

print(f"Video     : {frame_w}x{frame_h}  @  {video_info.fps} FPS")
print(f"Split line: Y={SPLIT_Y}px  (ignore top {ROI_SPLIT_RATIO*100:.0f}%)")
print(f"Debug mode: {'ON  — all overlays visible' if DEBUG else 'OFF — clean bbox output'}")

# ─────────────────────────────────────────
#  ④ HELPERS
# ─────────────────────────────────────────

def filter_detections_by_road(detections, road_mask, overlap_threshold=0.3):
    """Keep only boxes that overlap sufficiently with the road mask."""
    if len(detections) == 0:
        return detections
    keep = []
    for i, box in enumerate(detections.xyxy):
        x1, y1, x2, y2 = map(int, box)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(road_mask.shape[1], x2), min(road_mask.shape[0], y2)
        if x2 <= x1 or y2 <= y1:
            continue
        crop = road_mask[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        overlap = np.sum(crop == 1) / crop.size
        if overlap >= overlap_threshold:
            keep.append(i)
    if not keep:
        return sv.Detections.empty()
    return sv.Detections(
        xyxy       = detections.xyxy[keep],
        confidence = detections.confidence[keep] if detections.confidence is not None else None,
        class_id   = detections.class_id[keep]   if detections.class_id   is not None else None,
    )


def filter_detections_by_roi(detections, split_y):
    """Drop any detection whose bounding-box centre is above split_y."""
    if len(detections) == 0:
        return detections
    keep = [i for i, box in enumerate(detections.xyxy)
            if (box[1] + box[3]) / 2 >= split_y]
    if not keep:
        return sv.Detections.empty()
    return sv.Detections(
        xyxy       = detections.xyxy[keep],
        confidence = detections.confidence[keep] if detections.confidence is not None else None,
        class_id   = detections.class_id[keep]   if detections.class_id   is not None else None,
    )


def draw_roi_divider(frame_rgb, split_y, w, h):
    """
    DEBUG overlay: parking-sensor-style split line.
    - Translucent dark overlay on the ignored upper zone
    - Dashed yellow boundary line
    - Downward chevrons (▼) along the line
    - Zone labels
    """
    out = frame_rgb.copy()

    # Dim the ignored upper zone
    overlay = out.copy()
    cv2.rectangle(overlay, (0, 0), (w, split_y), (30, 30, 30), -1)
    out = cv2.addWeighted(overlay, 0.38, out, 0.62, 0)

    # Dashed boundary line
    dash, gap, x = 30, 15, 0
    while x < w:
        cv2.line(out, (x, split_y), (min(x + dash, w), split_y), (255, 220, 0), 2)
        x += dash + gap

    # Zone labels
    cv2.putText(out, "[ FAR-FIELD \u2014 IGNORED ]",
                (10, max(18, split_y - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (160, 160, 160), 1, cv2.LINE_AA)
    cv2.putText(out, "[ ACTIVE DETECTION ZONE ]",
                (10, split_y + 18),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 220, 0), 1, cv2.LINE_AA)

    # Parking-sensor chevrons ▼
    ch, cw = 16, 24
    for n in range(1, 6):
        cx = n * (w // 6)
        pts = np.array([
            [cx - cw // 2, split_y - ch // 2],
            [cx + cw // 2, split_y - ch // 2],
            [cx,           split_y + ch // 2]
        ], dtype=np.int32)
        cv2.fillPoly(out, [pts], (255, 220, 0))

    return out


def get_severity(normalized_area, edge_density):
    score = 0.6 * normalized_area + 0.4 * edge_density
    if score < 0.03:
        return "Minor",    (0, 255, 0),   score
    elif score < 0.08:
        return "Moderate", (255, 255, 0), score
    else:
        return "Severe",   (255, 0, 0),   score


# ─────────────────────────────────────────
#  ⑤ MAIN LOOP
# ─────────────────────────────────────────
current_road_mask = None
image_area        = frame_h * frame_w

with sv.VideoSink(target_path=TARGET_VIDEO_PATH, video_info=video_info) as sink:

    for i, frame in enumerate(tqdm(generator,
                                   total=video_info.total_frames,
                                   desc="RoadEye Inference")):

        rgb_frame      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image      = Image.fromarray(rgb_frame)
        annotated_frame = rgb_frame.copy()

        # ── A. ROAD SEGMENTATION ──────────────────────────────────────
        if i % MASK_UPDATE_FREQ == 0 or current_road_mask is None:
            current_road_mask = get_road_mask(pil_image, processor, seg_model, DEVICE)

        if DEBUG:
            # Green tint over detected road surface
            color_mask = np.zeros_like(annotated_frame)
            color_mask[current_road_mask == 1] = [0, 255, 0]
            annotated_frame = cv2.addWeighted(annotated_frame, 1.0, color_mask, 0.15, 0)

        # ── B. ROI DIVIDER (debug visual) ─────────────────────────────
        if DEBUG:
            annotated_frame = draw_roi_divider(annotated_frame, SPLIT_Y, frame_w, frame_h)

        # ── C. POTHOLE DETECTION ──────────────────────────────────────
        raw_detections = det_model.predict(pil_image, threshold=CONF_THRESHOLD)

        # ── D. FILTER: road area → then ROI zone ─────────────────────
        filtered_detections = filter_detections_by_road(
            raw_detections, current_road_mask, overlap_threshold=0.3
        )
        filtered_detections = filter_detections_by_roi(filtered_detections, SPLIT_Y)

        # ── E. PER-POTHOLE ANNOTATION ─────────────────────────────────
        for box, conf in zip(filtered_detections.xyxy, filtered_detections.confidence):

            x1, y1, x2, y2 = map(int, box)
            x1, y1 = max(0, x1),         max(0, y1)
            x2, y2 = min(frame_w, x2),   min(frame_h, y2)

            pothole_crop = rgb_frame[y1:y2, x1:x2]
            if pothole_crop.size == 0:
                continue

            # Edge detection (always computed, displayed only in DEBUG)
            gray         = cv2.cvtColor(pothole_crop, cv2.COLOR_RGB2GRAY)
            edges        = cv2.Canny(gray, 50, 150)
            edge_density = np.sum(edges > 0) / edges.size

            # Size metric
            box_area        = (x2 - x1) * (y2 - y1)
            normalized_area = box_area / image_area

            # Severity
            severity_label, color, severity_score = get_severity(normalized_area, edge_density)

            # ── [DEBUG] translucent filled box ────────────────────────
            if DEBUG:
                overlay = annotated_frame.copy()
                cv2.rectangle(overlay, (x1, y1), (x2, y2), color, -1)
                annotated_frame = cv2.addWeighted(overlay, 0.35, annotated_frame, 0.65, 0)

            # ── [DEBUG] white edge overlay inside box ─────────────────
            if DEBUG:
                region = annotated_frame[y1:y2, x1:x2]
                region[edges > 0] = [255, 255, 255]
                annotated_frame[y1:y2, x1:x2] = region

            # ── BOUNDING BOX (always drawn) ────────────────────────────
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)

            # ── LABEL BAR + TEXT (always drawn) ───────────────────────
            label   = f"{severity_label} ({conf:.2f})"
            lbl_y1  = max(0, y1 - 25)
            cv2.rectangle(annotated_frame, (x1, lbl_y1), (x2, y1), color, -1)
            cv2.putText(
                annotated_frame, label,
                (x1 + 5, max(15, y1 - 7)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                (0, 0, 0), 1, cv2.LINE_AA
            )

            # ── [DEBUG] stats HUD (edge density + score) ──────────────
            if DEBUG:
                hud = f"e={edge_density:.3f}  s={severity_score:.4f}"
                cv2.putText(
                    annotated_frame, hud,
                    (x1 + 3, min(frame_h - 5, y2 + 14)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.38,
                    (200, 200, 200), 1, cv2.LINE_AA
                )

        # ── F. WRITE FRAME ────────────────────────────────────────────
        output_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR)
        sink.write_frame(frame=output_frame)

print(f"\n✅ Done!  →  {TARGET_VIDEO_PATH}")

Video     : 1080x1920  @  30 FPS
Split line: Y=960px  (ignore top 50%)
Debug mode: ON  — all overlays visible


RoadEye Inference: 100%|██████████| 364/364 [01:14<00:00,  4.90it/s]


✅ Done!  →  /content/output/RoadEye_result.mp4
